# CUDA Attention Diagnostics + Fixed FFN Check

This notebook is intentionally diagnostic-only. It pulls the repo, checks that the latest CLI commands are available, runs the locked ActiveUnion FFN benchmark, and runs attention diagnostics. It does not start a pretraining lane or MacroV2.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any


def _run(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(str(x) for x in cmd), flush=True)
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail) > 120:
            tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError("Command failed with exit code " + str(rc) + "\n" + "".join(tail))


def _capture(cmd: list[str], *, cwd: Path | None = None) -> str:
    return subprocess.check_output(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        stderr=subprocess.STDOUT,
        text=True,
    )


def _checkout_repo_ref(repo: Path, repo_ref: str) -> Path:
    if not (repo / ".git").exists():
        return repo.resolve()
    if os.environ.get("SPEEDUP_THINGY_UPDATE", "1").strip().lower() in {"0", "false", "no", "off"}:
        return repo.resolve()
    _run(["git", "fetch", "origin"], cwd=repo)
    _run(["git", "checkout", repo_ref], cwd=repo)
    try:
        _run(["git", "pull", "--ff-only"], cwd=repo)
    except Exception:
        print("git pull skipped; likely checked out a detached commit/ref", flush=True)
    return repo.resolve()


def find_or_clone_repo() -> Path:
    repo_ref = os.environ.get("SPEEDUP_THINGY_REF", "master")
    env_root = os.environ.get("SPEEDUP_THINGY_REPO")
    if env_root:
        candidate = Path(env_root).expanduser()
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent):
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()
    clone_dir = Path(os.environ.get("SPEEDUP_THINGY_CLONE_DIR", "/content/Speedup-Thingy"))
    repo_url = os.environ.get("SPEEDUP_THINGY_REPO_URL", "https://github.com/BrownHujay/Speedup-Thingy.git")
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    if not clone_dir.exists():
        try:
            _run(["git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(clone_dir)])
        except Exception:
            _run(["git", "clone", repo_url, str(clone_dir)])
            _run(["git", "checkout", repo_ref], cwd=clone_dir)
    elif (clone_dir / "src" / "recursive_training_engine").exists():
        return _checkout_repo_ref(clone_dir, repo_ref)
    else:
        raise RuntimeError(f"Clone directory exists but is not a Speedup-Thingy repo: {clone_dir}")
    return clone_dir.resolve()


repo_root = find_or_clone_repo()
src_path = repo_root / "src"
os.environ["PYTHONPATH"] = str(src_path) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    import yaml  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

try:
    import recursive_training_engine  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root), "--no-deps"])

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA Colab runtime before running this notebook.")

torch.backends.cuda.matmul.allow_tf32 = True


def rte_cmd(*args: object) -> list[str]:
    return [sys.executable, "-c", "from recursive_training_engine.cli import main; main()", *[str(a) for a in args]]


def _extract_last_json(text: str) -> Any:
    decoder = json.JSONDecoder()
    best = None
    for idx, ch in enumerate(text):
        if ch not in "[{":
            continue
        try:
            obj, end = decoder.raw_decode(text[idx:])
        except json.JSONDecodeError:
            continue
        if text[idx + end :].strip() == "":
            best = obj
    if best is None:
        raise RuntimeError("No JSON object found in command output")
    return best


def run_rte_json(label: str, *args: object) -> Any:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    log_path = REPORT_DIR / f"{label}.log"
    cmd = rte_cmd(*args)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    proc = subprocess.run([str(x) for x in cmd], cwd=str(repo_root), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path.write_text(proc.stdout)
    if proc.returncode != 0:
        print(proc.stdout[-6000:])
        raise RuntimeError(f"{label} failed with exit code {proc.returncode}; log: {log_path}")
    result = _extract_last_json(proc.stdout)
    print(f"wrote log: {log_path}")
    return result


help_text = _capture(rte_cmd("--help"), cwd=repo_root)
required_commands = [
    "attention-diagnostics",
    "benchmark-active-union-ffn-train",
    "benchmark-active-union-model-train-step",
]
missing = [name for name in required_commands if name not in help_text]
if missing:
    raise RuntimeError(
        "This checkout does not contain the latest diagnostic commands: "
        + ", ".join(missing)
        + "\nSet SPEEDUP_THINGY_REF to the branch/commit with the new attention diagnostics."
    )

print("repo_root:", repo_root)
print("torch:", torch.__version__)
print("cuda device:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))


In [ ]:
def _bool_env(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def _int_list_env(name: str, default: list[int]) -> list[int]:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return list(default)
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


RUN_STAMP = os.environ.get("RUN_STAMP", time.strftime("%Y%m%d_%H%M%S"))
RUN_ROOT = repo_root / "runs" / f"colab_attention_diag_fixed_ffn_{RUN_STAMP}"
REPORT_DIR = RUN_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = Path(os.environ.get("CONFIG_PATH", repo_root / "configs" / "tiny.yaml"))
DENSE_CHECKPOINT = os.environ.get("DENSE_CHECKPOINT", "").strip()

RUN_FFN_FIXED_BENCH = _bool_env("RUN_FFN_FIXED_BENCH", True)
RUN_RANDOM_FULL_MODEL_SPEED = _bool_env("RUN_RANDOM_FULL_MODEL_SPEED", True)
RUN_ATTENTION_DIAGNOSTICS = _bool_env("RUN_ATTENTION_DIAGNOSTICS", bool(DENSE_CHECKPOINT))
RUN_RANDOM_ATTENTION_DIAG = _bool_env("RUN_RANDOM_ATTENTION_DIAG", not bool(DENSE_CHECKPOINT))

FFN_BENCH_SIZES = os.environ.get("FFN_BENCH_SIZES", "2048x8192x1024")
FFN_BENCH_SIZES = [x.strip() for x in FFN_BENCH_SIZES.split(",") if x.strip()]
ACTIVE_MS = _int_list_env("ACTIVE_MS", [192, 320])
BENCH_ITERS = int(os.environ.get("BENCH_ITERS", "5"))
BENCH_WARMUP = int(os.environ.get("BENCH_WARMUP", "1"))

D_MODEL = int(os.environ.get("D_MODEL", "256"))
D_FF = int(os.environ.get("D_FF", "2048"))
N_LAYERS = int(os.environ.get("N_LAYERS", "6"))
N_HEADS = int(os.environ.get("N_HEADS", "8"))
VOCAB_SIZE = int(os.environ.get("VOCAB_SIZE", "2048"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "16"))
SEQ_LEN = int(os.environ.get("SEQ_LEN", "64"))

ACTIVE_UNION_CAPS = _int_list_env("ACTIVE_UNION_CAPS", [192, 320])
CALIBRATION_TOKENS = int(os.environ.get("CALIBRATION_TOKENS", "8192"))
SVD_RANK = int(os.environ.get("SVD_RANK", "64"))
CLUSTERS = int(os.environ.get("CLUSTERS", "16"))
CANDIDATE_M = int(os.environ.get("CANDIDATE_M", "192"))
CLUSTER_ITERS = int(os.environ.get("CLUSTER_ITERS", "4"))
BENCH_DTYPE = os.environ.get("BENCH_DTYPE", "fp16")
BENCH_LOSS = os.environ.get("BENCH_LOSS", "clamped_ce")
RANDOM_MATRIX_SCALE = float(os.environ.get("RANDOM_MATRIX_SCALE", "0.25"))

ATTN_RANKS = _int_list_env("ATTN_RANKS", [32, 64, 128, 192])
ATTN_KV_GROUPS = _int_list_env("ATTN_KV_GROUPS", [1, 2, 4])
ATTN_EVAL_BATCHES = int(os.environ.get("ATTN_EVAL_BATCHES", "4"))
ATTN_CALIBRATION_TOKENS = int(os.environ.get("ATTN_CALIBRATION_TOKENS", str(CALIBRATION_TOKENS)))
ATTN_DTYPE = os.environ.get("ATTN_DTYPE", "fp32")

settings = {
    "run_root": str(RUN_ROOT),
    "config": str(CONFIG_PATH),
    "dense_checkpoint": DENSE_CHECKPOINT or None,
    "ffn_bench_sizes": FFN_BENCH_SIZES,
    "active_ms": ACTIVE_MS,
    "active_union_caps": ACTIVE_UNION_CAPS,
    "random_full_model": {"d_model": D_MODEL, "d_ff": D_FF, "layers": N_LAYERS, "heads": N_HEADS, "batch_size": BATCH_SIZE, "seq_len": SEQ_LEN, "loss": BENCH_LOSS, "matrix_scale": RANDOM_MATRIX_SCALE},
    "attention": {"ranks": ATTN_RANKS, "kv_groups": ATTN_KV_GROUPS, "eval_batches": ATTN_EVAL_BATCHES, "calibration_tokens": ATTN_CALIBRATION_TOKENS},
}
print(json.dumps(settings, indent=2))


## Fixed FFN Component Benchmark

This checks the locked packed ActiveUnion FFN path. It is FFN-only, so it does not depend on old checkpoints.

In [ ]:
ffn_bench = None
if RUN_FFN_FIXED_BENCH:
    args: list[object] = [
        "benchmark-active-union-ffn-train",
        "--active-m", *ACTIVE_MS,
        "--iters", BENCH_ITERS,
        "--warmup", BENCH_WARMUP,
        "--cuda-graphs",
        "--triton-swiglu-backward",
        "--output", REPORT_DIR / "fixed_ffn_active_union_benchmark.json",
    ]
    for size in FFN_BENCH_SIZES:
        args.extend(["--size", size])
    ffn_bench = run_rte_json("fixed_ffn_active_union_benchmark", *args)
    for row in ffn_bench.get("rows", []):
        print({
            "size": row.get("size"),
            "active_m": row.get("active_m"),
            "packed_fused_total_speedup": row.get("packed_fused_total_speedup"),
            "packed_fused_no_scatter_speedup": row.get("packed_fused_no_scatter_speedup"),
        })
else:
    print("Skipping fixed FFN benchmark.")


## Random Full-Model Speed Probe

This is timing-only. Random fp16 logits can overflow, so it uses clamped benchmark CE by default. Real checkpoint runs should use normal CE.

In [ ]:
random_model_bench = None
if RUN_RANDOM_FULL_MODEL_SPEED:
    random_model_bench = run_rte_json(
        "random_full_model_active_union_speed",
        "benchmark-active-union-model-train-step",
        "--config", CONFIG_PATH,
        "--random-init",
        "--d-model", D_MODEL,
        "--d-ff", D_FF,
        "--heads", N_HEADS,
        "--layers", N_LAYERS,
        "--vocab-size", VOCAB_SIZE,
        "--batch-size", BATCH_SIZE,
        "--seq-len", SEQ_LEN,
        "--active-union-cap", *ACTIVE_UNION_CAPS,
        "--calibration-tokens", CALIBRATION_TOKENS,
        "--rank", min(SVD_RANK, D_MODEL, D_FF),
        "--clusters", CLUSTERS,
        "--candidate-m", CANDIDATE_M,
        "--cluster-iters", CLUSTER_ITERS,
        "--dtype", BENCH_DTYPE,
        "--benchmark-loss", BENCH_LOSS,
        "--random-init-matrix-scale", RANDOM_MATRIX_SCALE,
        "--iters", BENCH_ITERS,
        "--warmup", BENCH_WARMUP,
        "--component-iters", 1,
        "--component-warmup", 1,
        "--event-iters", 3,
        "--event-warmup", 1,
        "--triton-swiglu-backward",
        "--output", REPORT_DIR / "random_full_model_active_union_speed.json",
    )
    for row in random_model_bench.get("rows", []):
        ep = row.get("sparse_event_profile", {})
        print({
            "cap": row.get("active_union_cap"),
            "total_step_speedup": row.get("event_total_step_speedup"),
            "ffn_fwd_bwd_speedup": row.get("event_ffn_fwd_bwd_speedup"),
            "qkv_ms": ep.get("attention_qkv_projection_forward_ms"),
            "score_value_ms": ep.get("attention_score_value_forward_ms"),
            "o_proj_ms": ep.get("attention_output_projection_forward_ms"),
            "misc_ms": ep.get("misc_overhead_ms"),
        })
else:
    print("Skipping random full-model speed probe.")


## Attention Diagnostics

If `DENSE_CHECKPOINT` is set, this runs on that checkpoint. Otherwise it can run a random smoke diagnostic so the command path is still testable.

In [ ]:
attention_diag = None
if RUN_ATTENTION_DIAGNOSTICS and DENSE_CHECKPOINT:
    attention_diag = run_rte_json(
        "attention_diagnostics_checkpoint",
        "attention-diagnostics",
        "--config", CONFIG_PATH,
        "--dense-checkpoint", DENSE_CHECKPOINT,
        "--batch-size", BATCH_SIZE,
        "--seq-len", SEQ_LEN,
        "--calibration-tokens", ATTN_CALIBRATION_TOKENS,
        "--eval-batches", ATTN_EVAL_BATCHES,
        "--ranks", *ATTN_RANKS,
        "--kv-groups", *ATTN_KV_GROUPS,
        "--cluster-iters", CLUSTER_ITERS,
        "--dtype", ATTN_DTYPE,
        "--output", REPORT_DIR / "attention_diagnostics_checkpoint.json",
    )
elif RUN_RANDOM_ATTENTION_DIAG:
    attention_diag = run_rte_json(
        "attention_diagnostics_random",
        "attention-diagnostics",
        "--config", CONFIG_PATH,
        "--random-init",
        "--d-model", D_MODEL,
        "--d-ff", D_FF,
        "--heads", N_HEADS,
        "--layers", N_LAYERS,
        "--vocab-size", VOCAB_SIZE,
        "--batch-size", BATCH_SIZE,
        "--seq-len", SEQ_LEN,
        "--calibration-tokens", ATTN_CALIBRATION_TOKENS,
        "--eval-batches", ATTN_EVAL_BATCHES,
        "--ranks", *ATTN_RANKS,
        "--kv-groups", *ATTN_KV_GROUPS,
        "--cluster-iters", CLUSTER_ITERS,
        "--dtype", ATTN_DTYPE,
        "--output", REPORT_DIR / "attention_diagnostics_random.json",
    )
else:
    print("Skipping attention diagnostics. Set DENSE_CHECKPOINT or RUN_RANDOM_ATTENTION_DIAG=1.")

if attention_diag:
    print("KV rollout:")
    for row in attention_diag.get("kv_group_rollout", []):
        print(row)
    print("Low-rank rollout:")
    for row in attention_diag.get("lowrank_rollout", []):
        print(row)


In [ ]:
summary = {
    "run_root": str(RUN_ROOT),
    "fixed_ffn_report": str(REPORT_DIR / "fixed_ffn_active_union_benchmark.json") if ffn_bench else None,
    "random_full_model_report": str(REPORT_DIR / "random_full_model_active_union_speed.json") if random_model_bench else None,
    "attention_report": str(REPORT_DIR / ("attention_diagnostics_checkpoint.json" if DENSE_CHECKPOINT else "attention_diagnostics_random.json")) if attention_diag else None,
}
print(json.dumps(summary, indent=2))
